# Figure 2
Copy of Figure 2 for font-size tuning and export-size experiments.

In [31]:
# --- Setup: libraries and config ---
library(tidyverse)
library(arrow)
library(patchwork)
library(cowplot)
library(jsonlite)
library(fs)

config_candidates <- c(
  Sys.getenv("CONFIG_PATH", unset = ""),
  fs::path(".", "config.json"),
  fs::path("..", "config.json"),
  fs::path("..", "..", "config.json")
)
config_candidates <- normalizePath(unique(config_candidates[nzchar(config_candidates)]), winslash = "/", mustWork = FALSE)
config_path <- config_candidates[file.exists(config_candidates)][1]
if (is.na(config_path) || !nzchar(config_path)) {
  stop("Could not locate config.json. Set CONFIG_PATH or run from the project tree.")
}
config <- jsonlite::fromJSON(config_path)
project_root <- normalizePath(config$project_root, winslash = "/", mustWork = FALSE)
source(fs::path(project_root, "scripts", "utils", "plot_style.R"))

# --- Theme and typography ---
plot_style <- get_plot_style(config)
plot_style$axis_line_width <- 0.20
plot_style$axis_tick_width <- 0.20
plot_style$axis_tick_length_pt <- 1.8
font_family_use <- get_export_font_family()
plot_style$font_family <- font_family_use

# Fixed typography for 88 mm assembled figure output.
axis_title_pt <- 5
axis_text_pt <- 5
plot_title_pt <- 6
legend_title_pt <- 5
legend_text_pt <- 5
panel_tag_pt <- 6

theme_pub <- make_theme_pub(
  style = plot_style,
  axis_title_pt = axis_title_pt,
  axis_text_pt = axis_text_pt,
  plot_title_pt = plot_title_pt,
  legend_title_pt = legend_title_pt,
  legend_text_pt = legend_text_pt,
  legend_position = 'none',
  base_size_pt = 5
) +
  theme(
    plot.title = element_text(face = 'plain', margin = margin(0, 0, 0, 0)),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
    plot.margin = margin(0, 0, 0, 0)
  )

top_strip_ratio <- 0.22
right_strip_ratio <- 0.16
box_right_strip_ratio <- 0.00
box_top_strip_ratio <- 0.08
vendor_order <- c("GE", "Philips", "Siemens")
figure2_dir <- fs::path(project_root, "figures", "Figure2")
fs::dir_create(fs::path(figure2_dir, "panels"), recurse = TRUE)

# Target final assembled Figure 2 size (for Illustrator assembly).
assembled_width_mm <- get_figure_dims(layout = 'one_column', height_mm = 100)$width_mm
base_figure_dims <- get_figure_dims(layout = 'two_column', height_mm = 160)
assembled_height_mm <- base_figure_dims$height_mm * (assembled_width_mm / base_figure_dims$width_mm)
assembled_figure_dims <- list(width_mm = assembled_width_mm, height_mm = assembled_height_mm)
panel_cell_dims <- list(width_mm = assembled_figure_dims$width_mm / 2, height_mm = assembled_figure_dims$height_mm / 2)
# In the combined layout, panel D occupies half of its right-hand cell (the other half is spacer).
panel_d_dims_final <- list(width_mm = panel_cell_dims$width_mm / 2, height_mm = panel_cell_dims$height_mm)

# --- Data ---
data_candidates <- c(
  file.path(project_root, 'data', 'harmonized_data', 'merged_data_meisler_analyses_harmonized.parquet'),
  file.path(project_root, 'data', 'raw_data', 'merged_data_meisler_analyses.parquet')
)
data_path <- data_candidates[file.exists(data_candidates)][1]
if (is.na(data_path) || !nzchar(data_path)) stop('No parquet data file found in expected locations.')
df <- read_parquet(data_path) %>%
  as_tibble() %>%
  mutate(scanner_manufacturer = factor(as.character(scanner_manufacturer), levels = vendor_order))

required_cols <- c(
  'scanner_manufacturer',
  'raw_neighbor_corr', 't1post_neighbor_corr',
  'raw_dwi_contrast', 't1post_dwi_contrast',
  'CNR0_median', 'CNR1_median', 'CNR2_median', 'CNR3_median', 'CNR4_median'
)
missing_cols <- setdiff(required_cols, names(df))
if (length(missing_cols) > 0) stop(paste('Missing required columns:', paste(missing_cols, collapse = ', ')))

# --- Export helper (mm-based PDF for this figure) ---
save_pdf_arial <- function(path, plot, width_mm, height_mm) {
  ggsave(
    filename = path,
    plot = plot,
    width = width_mm,
    height = height_mm,
    units = 'mm',
    device = cairo_pdf,
    family = font_family_use
  )
}

# --- Panel helper functions ---
make_title_plot <- function(title_text) {
  ggplot() +
    labs(title = title_text) +
    theme_void(base_family = plot_style$font_family) +
    theme(
      plot.title = element_text(size = plot_title_pt, face = 'plain', hjust = 0.5, margin = margin(0, 0, 0, 0)),
      plot.margin = margin(0, 0, 0, 0)
    )
}

add_panel_tag <- function(panel_plot, tag_text) {
  tag_layer <- ggplot() +
    annotate(
      'text',
      x = 0.01,
      y = 0.995,
      label = tag_text,
      hjust = 0,
      vjust = 1,
      family = plot_style$font_family,
      fontface = 'bold',
      size = panel_tag_pt / .pt
    ) +
    coord_cartesian(xlim = c(0, 1), ylim = c(0, 1), expand = FALSE, clip = 'off') +
    theme_void() +
    theme(plot.margin = margin(0, 0, 0, 0))

  cowplot::ggdraw() +
    cowplot::draw_plot(panel_plot, 0, 0, 1, 1) +
    cowplot::draw_plot(tag_layer, 0, 0, 1, 1)
}

build_scatter_with_marginals <- function(data, xvar, yvar, title_text, xlab, ylab, fixed_upper = NULL) {
  dd <- data %>%
    select(scanner_manufacturer, x = all_of(xvar), y = all_of(yvar)) %>%
    filter(!is.na(scanner_manufacturer), !is.na(x), !is.na(y))

  if (nrow(dd) == 0) stop(paste('No data for', title_text))

  xlim <- range(dd$x, na.rm = TRUE)
  ylim <- range(dd$y, na.rm = TRUE)
  if (!is.null(fixed_upper)) {
    xlim[2] <- fixed_upper
    ylim[2] <- fixed_upper
  }

  # Single strip size used in both directions so top/right marginals match.
  strip_ratio <- 0.12
  title_ratio <- 0.085

  p_scatter <- ggplot(dd, aes(x = x, y = y, color = scanner_manufacturer)) +
    geom_abline(intercept = 0, slope = 1, linetype = 'dotted', linewidth = 0.28, color = 'black') +
    geom_point(alpha = 0.35, size = 0.55, stroke = 0) +
    scale_color_manual(values = vendor_colors, limits = vendor_order, drop = FALSE, guide = 'none') +
    coord_cartesian(xlim = xlim, ylim = ylim, expand = FALSE) +
    labs(x = xlab, y = ylab) +
    theme_pub +
    theme(
      axis.line = element_blank(),
      panel.border = element_rect(color = 'black', fill = NA, linewidth = 0.20),
      aspect.ratio = 1,
      legend.position = 'none',
      plot.margin = margin(0, 0, 0, 0)
    )

  p_top <- ggplot(dd, aes(x = x, y = after_stat(ifelse(scaled < 0.015, NA_real_, scaled)), color = scanner_manufacturer, fill = scanner_manufacturer)) +
    geom_density(alpha = 0.20, linewidth = 0.28) +
    scale_color_manual(values = vendor_colors, limits = vendor_order, drop = FALSE, guide = 'none') +
    scale_fill_manual(values = vendor_colors, limits = vendor_order, drop = FALSE, guide = 'none') +
    scale_y_continuous(limits = c(0, 1.02), breaks = c(0.75, 0.85, 0.95), labels = c('0.75', '0.85', '0.95'), expand = expansion(mult = c(0, 0))) +
    coord_cartesian(xlim = xlim, expand = FALSE) +
    labs(y = 'Preprocessed') +
    theme_pub +
    theme(
      axis.title.x = element_blank(),
      axis.text.x = element_blank(),
      axis.ticks.x = element_blank(),
      axis.line.x = element_blank(),
      axis.title.y = element_text(color = NA),
      axis.text.y = element_text(color = NA),
      axis.ticks.y = element_line(color = NA),
      axis.line.y = element_blank(),
      panel.border = element_blank(),
      legend.position = 'none',
      plot.margin = margin(0, 0, -2, 0)
    )

  p_right <- ggplot(dd, aes(x = y, y = after_stat(ifelse(scaled < 0.015, NA_real_, scaled)), color = scanner_manufacturer, fill = scanner_manufacturer)) +
    geom_density(alpha = 0.20, linewidth = 0.28) +
    scale_color_manual(values = vendor_colors, limits = vendor_order, drop = FALSE, guide = 'none') +
    scale_fill_manual(values = vendor_colors, limits = vendor_order, drop = FALSE, guide = 'none') +
    scale_y_continuous(limits = c(0, 1.02), expand = expansion(mult = c(0, 0))) +
    coord_flip(xlim = ylim, expand = FALSE) +
    theme_void(base_family = plot_style$font_family) +
    theme(
      legend.position = 'none',
      plot.margin = margin(0, 0, 0, -2)
    )

  p_blank <- ggplot() +
    theme_void() +
    theme(plot.margin = margin(0, 0, 0, 0))

  top_row <- cowplot::plot_grid(
    p_top,
    p_blank,
    nrow = 1,
    align = 'h',
    axis = 'tb',
    rel_widths = c(1, strip_ratio)
  )

  bottom_row <- cowplot::plot_grid(
    p_scatter,
    p_right,
    nrow = 1,
    align = 'h',
    axis = 'tb',
    rel_widths = c(1, strip_ratio)
  )

  main_block <- cowplot::plot_grid(
    top_row,
    bottom_row,
    ncol = 1,
    align = 'v',
    axis = 'lr',
    rel_heights = c(strip_ratio, 1)
  )

  title_plot <- ggplot() +
    labs(title = title_text) +
    theme_void(base_family = plot_style$font_family) +
    theme(
      plot.title = element_text(
        size = plot_title_pt,
        face = 'plain',
        hjust = 0.5,
        family = plot_style$font_family,
        margin = margin(0, 0, -5, 0)
      ),
      plot.margin = margin(0, 0, 0, 0)
    )

  cowplot::plot_grid(
    title_plot,
    main_block,
    ncol = 1,
    rel_heights = c(title_ratio, 1)
  )
}

build_box_panel <- function(main_plot, title_text) {
  (make_title_plot(title_text) / main_plot) +
    plot_layout(heights = c(0.04, 1), guides = 'keep') &
    theme(legend.position = 'none', plot.margin = margin(0, 0, 0, 0))
}


## Panels A and B
Scatter plots comparing raw vs preprocessed QC metrics by scanner vendor, each with top/right marginal densities.

In [32]:
# --- Panels A & B: scatter with marginals + export ---
p_a <- build_scatter_with_marginals(
  data = df,
  xvar = 'raw_neighbor_corr',
  yvar = 't1post_neighbor_corr',
  title_text = 'Neighboring dMRI Correlation',
  xlab = 'Raw',
  ylab = 'Preprocessed',
  fixed_upper = 1
)

p_b <- build_scatter_with_marginals(
  data = df,
  xvar = 'raw_dwi_contrast',
  yvar = 't1post_dwi_contrast',
  title_text = 'dMRI Contrast',
  xlab = 'Raw',
  ylab = 'Preprocessed'
)


panel_dims <- panel_cell_dims
save_pdf_arial(fs::path(figure2_dir, "panels", "Figure2_panel_a.pdf"), p_a, width_mm = panel_dims$width_mm, height_mm = panel_dims$height_mm)
ggsave(fs::path(figure2_dir, "panels", "Figure2_panel_a.jpg"), p_a, width = panel_dims$width_mm, height = panel_dims$height_mm, units = 'mm', dpi = 600, device = 'jpeg')
save_pdf_arial(fs::path(figure2_dir, "panels", "Figure2_panel_b.pdf"), p_b, width_mm = panel_dims$width_mm, height_mm = panel_dims$height_mm)
ggsave(fs::path(figure2_dir, "panels", "Figure2_panel_b.jpg"), p_b, width = panel_dims$width_mm, height = panel_dims$height_mm, units = 'mm', dpi = 600, device = 'jpeg')


Warning message:
“Removed 1088 rows containing missing values or values outside the scale range
(`geom_density()`).”
Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
“font family 'Arial' not found in PostScript font database”
Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
“font family 'Arial' not found in PostScript font database”
Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
“font family 'Arial' not found in PostScript font database”
Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
“font family 'Arial' not found in PostScript font database”
Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
“font family 'Arial' not found in PostScript font database”
Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
“font family 'Arial' not found in PostScript font database”
Warning message in grid.Call(C_

## Panel C
Vendor-stratified boxplots of median CNR across diffusion shells (b=500, 1000, 2000, 3000).

In [33]:
cnr_long <- df %>%
  select(scanner_manufacturer, CNR1_median, CNR2_median, CNR3_median, CNR4_median) %>%
  filter(scanner_manufacturer %in% vendor_order) %>%
  pivot_longer(cols = starts_with('CNR'), names_to = 'shell', values_to = 'value') %>%
  filter(!is.na(value)) %>%
  mutate(
    scanner_manufacturer = factor(scanner_manufacturer, levels = vendor_order),
    shell = factor(shell, levels = c('CNR1_median', 'CNR2_median', 'CNR3_median', 'CNR4_median'))
  )

cnr_stats <- cnr_long %>%
  group_by(shell, scanner_manufacturer) %>%
  summarise(
    ymin = quantile(value, 0.10, na.rm = TRUE),
    lower = quantile(value, 0.25, na.rm = TRUE),
    middle = quantile(value, 0.50, na.rm = TRUE),
    upper = quantile(value, 0.75, na.rm = TRUE),
    ymax = quantile(value, 0.90, na.rm = TRUE),
    .groups = 'drop'
  )

shell_labels <- c(
  CNR1_median = 'italic(b) == 500',
  CNR2_median = 'italic(b) == 1000',
  CNR3_median = 'italic(b) == 2000',
  CNR4_median = 'italic(b) == 3000'
)

p_c_main <- ggplot(cnr_stats, aes(x = shell, fill = scanner_manufacturer)) +
  geom_boxplot(
    aes(ymin = ymin, lower = lower, middle = middle, upper = upper, ymax = ymax),
    stat = 'identity',
    position = position_dodge2(width = 0.80, preserve = 'single'),
    width = 0.65,
    color = 'black',
    linewidth = 0.20,
    whisker.linewidth = 0.20,
    median.linewidth = 0.28,
    outlier.shape = NA
  ) +
  scale_fill_manual(values = vendor_colors, limits = vendor_order, drop = FALSE, guide = 'none') +
  scale_x_discrete(labels = function(x) parse(text = unname(shell_labels[x])), drop = FALSE) +
  labs(x = 'dMRI Shell', y = 'Median CNR') +
  theme_pub +
  theme(
    legend.position = 'none',
    aspect.ratio = 1,
    axis.text.x = element_text(angle = 32, hjust = 1, vjust = 1)
  )

p_c <- build_box_panel(p_c_main, 'CNR Distribution by dMRI Shell')


panel_dims <- panel_cell_dims
save_pdf_arial(fs::path(figure2_dir, "panels", "Figure2_panel_c.pdf"), p_c, width_mm = panel_dims$width_mm, height_mm = panel_dims$height_mm)
ggsave(fs::path(figure2_dir, "panels", "Figure2_panel_c.jpg"), p_c, width = panel_dims$width_mm, height = panel_dims$height_mm, units = 'mm', dpi = 600, device = 'jpeg')


## Panel D
Vendor-stratified boxplots of median b=0 shell tSNR.

In [34]:
tsnr_stats <- df %>%
  select(scanner_manufacturer, CNR0_median) %>%
  filter(scanner_manufacturer %in% vendor_order, !is.na(CNR0_median)) %>%
  mutate(scanner_manufacturer = factor(scanner_manufacturer, levels = vendor_order)) %>%
  group_by(scanner_manufacturer) %>%
  summarise(
    ymin = quantile(CNR0_median, 0.10, na.rm = TRUE),
    lower = quantile(CNR0_median, 0.25, na.rm = TRUE),
    middle = quantile(CNR0_median, 0.50, na.rm = TRUE),
    upper = quantile(CNR0_median, 0.75, na.rm = TRUE),
    ymax = quantile(CNR0_median, 0.90, na.rm = TRUE),
    .groups = 'drop'
  )

p_d_main <- ggplot(tsnr_stats, aes(x = scanner_manufacturer, fill = scanner_manufacturer)) +
  geom_boxplot(
    aes(ymin = ymin, lower = lower, middle = middle, upper = upper, ymax = ymax),
    stat = 'identity',
    width = 0.62,
    color = 'black',
    linewidth = 0.20,
    whisker.linewidth = 0.20,
    median.linewidth = 0.28,
    outlier.shape = NA
  ) +
  scale_fill_manual(values = vendor_colors, limits = vendor_order, drop = FALSE, guide = 'none') +
  scale_x_discrete(limits = vendor_order, drop = FALSE, expand = expansion(mult = c(0.34, 0.08))) +
  scale_y_continuous(limits = c(0, 45), expand = expansion(mult = c(0, 0))) +
  labs(
    x = 'Scanner Vendor',
    y = 'Median tSNR'
  ) +
  theme_pub +
  theme(
    legend.position = 'none',
    axis.text.x = element_text(angle = 35, hjust = 1, vjust = 1),
    aspect.ratio = 2
  )

p_d <- build_box_panel(
  p_d_main,
  expression(tSNR~of~the~italic(b)==0~Shell)
)


panel_d_dims <- panel_d_dims_final
save_pdf_arial(fs::path(figure2_dir, "panels", "Figure2_panel_d.pdf"), p_d, width_mm = panel_d_dims$width_mm, height_mm = panel_d_dims$height_mm)
ggsave(fs::path(figure2_dir, "panels", "Figure2_panel_d.jpg"), p_d, width = panel_d_dims$width_mm, height_mm = panel_d_dims$height_mm, units = "mm", dpi = 600, device = "jpeg")


Saving 22 x 169 mm image


ERROR: Error in f(...): unused argument (height_mm = 39.1111111111111)


## Figure Assembly
Combine panels A-D into the final two-column Figure 2 layout and export publication files.

In [ ]:
panel_d_cell <- p_d | plot_spacer()

top_row <- cowplot::plot_grid(
  p_a,
  p_b,
  nrow = 1,
  align = 'h',
  axis = 'tb',
  labels = c('a', 'b'),
  label_fontfamily = font_family_use,
  label_fontface = 'bold',
  label_size = panel_tag_pt,
  label_x = 0.01,
  label_y = 0.995,
  hjust = 0,
  vjust = 1
)

bottom_row <- cowplot::plot_grid(
  p_c,
  panel_d_cell,
  nrow = 1,
  align = 'h',
  axis = 'tb',
  rel_widths = c(1, 1),
  labels = c('c', 'd'),
  label_fontfamily = font_family_use,
  label_fontface = 'bold',
  label_size = panel_tag_pt,
  label_x = 0.01,
  label_y = 0.995,
  hjust = 0,
  vjust = 1
)

fig2 <- cowplot::plot_grid(
  top_row,
  bottom_row,
  ncol = 1,
  rel_heights = c(1, 1),
  align = 'v',
  axis = 'lr'
)

fig2

# Export full 4-panel figure at one-column width (88 mm).
dims <- assembled_figure_dims

out_pdf <- fs::path(figure2_dir, "Figure2_combined.pdf")
out_png <- fs::path(figure2_dir, "Figure2_combined.png")

save_pdf_arial(out_pdf, fig2, width_mm = dims$width_mm, height_mm = dims$height_mm)
ggsave(out_png, fig2, width = dims$width_mm, height = dims$height_mm, units = 'mm', dpi = 600)

cat('Saved:\n', out_pdf, '\n', out_png, '\n', sep = '')


Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
“font family 'Arial' not found in PostScript font database”
Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
“font family 'Arial' not found in PostScript font database”
Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
“font family 'Arial' not found in PostScript font database”
Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
“font CMap for family 'Arial' not found in font database”


ERROR: Error in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, : failed to find or load PDF CID font
